# Grand Canonical Monte Carlo (GCMC): 2D Patchy Particles

Implements the **μVT ensemble** for 2-patch patchy particles in a periodic 2D box.
Unlike the NVT simulation in `run_yield_simulation_cluster.py`, **N is not fixed**:
insertion and deletion MC moves exchange particles with an implicit reservoir at
chemical potential μ.

---
## How to choose μ

### 1. Ideal-gas starting point (Λ = 1, 2D)

$$\mu_{\rm ideal} = k_BT \ln(\rho_N), \qquad \rho_N = \frac{N}{A} = \frac{\phi}{\pi r^2}$$

For φ = 0.2, r = 1: ρ_N ≈ 0.064 → **μ_ideal ≈ −2.75 k_BT**.

### 2. Excess contribution from attractive bonds

Each Morse bond contributes roughly −ε/2 per particle to the excess chemical potential:
$$\mu = \mu_{\rm ideal} + \mu_{\rm exc}, \qquad \mu_{\rm exc} \approx -z_{\rm bonds}\,\varepsilon/2 < 0$$

With E_AB = 5 and ~1–2 bonds per particle: μ* ≈ −6 to −8 k_BT.

### 3. Practical: μ-scan

Run short GCMC sweeps at a grid of μ values, plot ⟨N⟩ vs μ, pick the value giving ⟨N⟩ ≈ N_target.
Bonus: the **Widom test-particle insertion** cell at the end measures μ_exc directly.

---
### Acceptance criteria

$$P_{\rm acc}^{\rm disp} = \min\!\left(1,\; e^{-\Delta U/k_BT}\right)$$

$$P_{\rm acc}^{\rm ins} = \min\!\left(1,\; \frac{A}{N+1}\,e^{(\mu-\Delta U_{\rm ins})/k_BT}\right)$$

$$P_{\rm acc}^{\rm del} = \min\!\left(1,\; \frac{N}{A}\,e^{(-\mu+E_i)/k_BT}\right)$$

where ΔU_ins is the energy of the trial particle with the existing N particles,
and E_i is the energy of the deleted particle with the remaining N−1 particles.
(Thermal wavelength Λ = 1 is absorbed into the reference state of μ.)

## 1 — Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import Circle
import sys
import os

sys.path.insert(0, os.path.abspath('../..'))

from base.config import PATCH_SIZE, ALPHA

PATCH_SIGMA = float(PATCH_SIZE)
ALPHA_VAL   = float(ALPHA)
print(f'PATCH_SIGMA = {PATCH_SIGMA:.4f}')
print(f'ALPHA       = {ALPHA_VAL}')

## 2 — Simulation Parameters

In [ ]:
# ── Box ─────────────────────────────────────────────────────────────────────
L = 20.0          # side length; particles in [-L/2, L/2]^2
A = L**2          # area = 1600

# ── Target density and initial N ─────────────────────────────────────────────
phi_target = 0.2
r_center   = 1.0
N_init     = int(phi_target * A / (np.pi * r_center**2))   # ~ 102

# ── Temperature ──────────────────────────────────────────────────────────────
kT = 1.0

# ── Patch geometry ───────────────────────────────────────────────────────────
opening_angle = np.pi / 2   # 90 deg between patch A and patch B (square assembly)
r_patch       = 1.0         # arm length = CENTER_RADIUS

# ── Interaction energies ─────────────────────────────────────────────────────
E_AA  = 1.0
E_BB  = 1.0
E_AB  = 5.0      # strong cross-patch attraction
rep_A = 10_000.0     # soft-sphere prefactor

# ── Step-size parameters for displacement moves ───────────────────────────────
DR_MAX     = 0.05   # max translation step
DTHETA_MAX = 0.05   # max rotation step (rad)

params = dict(
    opening_angle = opening_angle,
    r_patch       = r_patch,
    E_AA          = E_AA,
    E_BB          = E_BB,
    E_AB          = E_AB,
    rep_A         = rep_A,
    alpha         = ALPHA_VAL,
    patch_sigma   = PATCH_SIGMA,
)

rho_N     = phi_target / (np.pi * r_center**2)
mu_ideal  = kT * np.log(rho_N)
 
print(f'Box:         L = {L},  A = {A:.0f}')
print(f'Target:      phi = {phi_target}  =>  N_init = {N_init}')
print(f'Energies:    E_AA={E_AA}, E_BB={E_BB}, E_AB={E_AB}  (zeta={E_AB/(E_AA+E_BB):.1f})')
print(f'mu_ideal = kT*ln(rho_N) = {mu_ideal:.3f}  (scan range roughly [-12, -4])')

## 3 — Energy Functions (numpy, periodic boundary conditions)

All distances use the **minimum-image convention**.  
Incremental `particle_energy(i, q, ...)` is O(N) — used for every MC move.

In [ ]:
def minimum_image(dr, L):
    """Minimum-image displacement. Works for any shape of dr."""
    return dr - L * np.round(dr / L)


def _smooth_morse_weight(r, patch_sigma):
    """
    Smooth multiplicative window matching jax-md's multiplicative_isotropic_cutoff
    with r_onset=0:  w(r) = (rc²-r²)²(rc²+2r²)/rc⁶
    w(0) = 1,  w(patch_sigma) = 0,  C¹ everywhere.
    Replaces the hard step  where(r < patch_sigma, 1, 0).
    """
    rc2 = patch_sigma ** 2
    r2  = r ** 2
    return np.where(r < patch_sigma,
                    (rc2 - r2) ** 2 * (rc2 + 2.0 * r2) / rc2 ** 3,
                    0.0)


def get_patches(q, r_patch, opening_angle):
    """
    q : (N, 3)  [x, y, theta]
    Returns pA, pB : (N, 2) patch positions in lab frame.
    Convention: theta = direction of patch A.
        pA_i = center_i + r_patch * [cos(theta_i),                  sin(theta_i)]
        pB_i = center_i + r_patch * [cos(theta_i + opening_angle),  sin(theta_i + opening_angle)]
    """
    c  = q[:, :2]
    th = q[:, 2]
    pA = c + r_patch * np.stack([np.cos(th),                  np.sin(th)],                  axis=-1)
    pB = c + r_patch * np.stack([np.cos(th + opening_angle),  np.sin(th + opening_angle)],  axis=-1)
    return pA, pB


def particle_energy(i, q, L, params):
    """
    Energy of particle i interacting with all j != i.  O(N).
    Used for incremental updates in every MC move.
    """
    N = len(q)
    if N <= 1:
        return 0.0

    mask = np.ones(N, dtype=bool)
    mask[i] = False
    q_j = q[mask]

    # ── center-center soft-sphere ─────────────────────────────────────────────
    ci       = q[i, :2]
    cj       = q_j[:, :2]
    dr_cc    = minimum_image(ci[None, :] - cj, L)          # (N-1, 2)
    r_cc     = np.sqrt(np.sum(dr_cc**2, axis=-1))           # (N-1,)
    sigma_cc = 2.0 * params['r_patch']
    v_rep    = np.where(r_cc < sigma_cc,
                        params['rep_A'] * (1.0 - r_cc / sigma_cc)**2,
                        0.0)

    # ── patch positions (theta = patch A direction) ───────────────────────────
    thi  = q[i, 2]
    thj  = q_j[:, 2]
    oa   = params['opening_angle']
    rp   = params['r_patch']
    pAi  = ci + rp * np.array([np.cos(thi),      np.sin(thi)])
    pBi  = ci + rp * np.array([np.cos(thi + oa), np.sin(thi + oa)])
    pAj  = cj + rp * np.stack([np.cos(thj),      np.sin(thj)],      axis=-1)
    pBj  = cj + rp * np.stack([np.cos(thj + oa), np.sin(thj + oa)], axis=-1)

    # ── patch-patch Morse with smooth cutoff ──────────────────────────────────
    def morse(pi, pj_arr, eps):
        dr = minimum_image(pi[None, :] - pj_arr, L)
        r  = np.sqrt(np.sum(dr**2, axis=-1))
        v  = eps * (1.0 - np.exp(-params['alpha'] * r))**2 - eps
        return _smooth_morse_weight(r, params['patch_sigma']) * v

    return float(np.sum(
        v_rep
        + morse(pAi, pAj, params['E_AA'])
        + morse(pBi, pBj, params['E_BB'])
        + morse(pAi, pBj, params['E_AB'])
        + morse(pBi, pAj, params['E_AB'])
    ))


def total_energy_pbc(q, L, params):
    """Full O(N^2) energy — for tracking/verification only."""
    if len(q) == 0:
        return 0.0
    return 0.5 * sum(particle_energy(i, q, L, params) for i in range(len(q)))


print('Energy functions defined.')
print('  Patch convention: theta = patch A direction  (pA at theta, pB at theta+alpha)')
print('  Morse cutoff: smooth window  w(r) = (rc²-r²)²(rc²+2r²)/rc⁶  (r_onset=0)')

## 4 — Initial Conditions (Random Sequential Addition)

In [ ]:
def generate_ic(N, L, params, seed=42, max_att=20000):
    """Place N non-overlapping particles by random sequential addition."""
    rng   = np.random.default_rng(seed)
    sigma = 2.0 * params['r_patch']
    pts   = []
    for n in range(N):
        for _ in range(max_att):
            x  = rng.uniform(-L/2, L/2)
            y  = rng.uniform(-L/2, L/2)
            th = rng.uniform(0.0, 2.0 * np.pi)
            ok = all(
                np.sqrt(np.sum(minimum_image(
                    np.array([x - p[0], y - p[1]]), L)**2)) >= sigma
                for p in pts
            )
            if ok:
                pts.append([x, y, th])
                break
        else:
            raise RuntimeError(f'RSA failed at particle {n+1}/{N}')
    return np.array(pts)


q0 = generate_ic(N_init, L, params, seed=42)
E0 = total_energy_pbc(q0, L, params)
print(f'IC:  N = {len(q0)},  E = {E0:.2f},  E/N = {E0/len(q0):.3f}')

## 5 — GCMC Move Functions and Sweep

In [ ]:
def mc_displacement(q, L, kT, params, rng):
    """Metropolis translation + rotation of one random particle."""
    N = len(q)
    if N == 0:
        return q, False
    i       = rng.integers(0, N)
    E_old   = particle_energy(i, q, L, params)
    q_new   = q.copy()
    q_new[i, 0] += rng.uniform(-DR_MAX,     DR_MAX)
    q_new[i, 1] += rng.uniform(-DR_MAX,     DR_MAX)
    q_new[i, 2] += rng.uniform(-DTHETA_MAX, DTHETA_MAX)
    q_new[i, :2]  = np.mod(q_new[i, :2] + L/2, L) - L/2   # wrap into box
    E_new   = particle_energy(i, q_new, L, params)
    if rng.random() < np.exp(-(E_new - E_old) / kT):
        return q_new, True
    return q, False


def mc_insertion(q, L, kT, mu, params, rng):
    """Grand-canonical insertion of a trial particle at a random position."""
    N     = len(q)
    x     = rng.uniform(-L/2, L/2)
    y     = rng.uniform(-L/2, L/2)
    th    = rng.uniform(0.0, 2.0 * np.pi)
    q_new = np.vstack([q, [[x, y, th]]])
    dE    = particle_energy(N, q_new, L, params)   # energy of new particle with existing N
    # acc = A/(N+1) * exp((mu - dE)/kT)
    acc   = (L**2 / (N + 1)) * np.exp((mu - dE) / kT)
    if rng.random() < min(1.0, acc):
        return q_new, True
    return q, False


def mc_deletion(q, L, kT, mu, params, rng):
    """Grand-canonical deletion of a random particle."""
    N = len(q)
    if N == 0:
        return q, False
    i     = rng.integers(0, N)
    E_i   = particle_energy(i, q, L, params)   # < 0 if particle is bonded
    q_new = np.delete(q, i, axis=0)
    # dE = U(N-1) - U(N) = -E_i
    # acc = N/A * exp((-mu - dE)/kT) = N/A * exp((-mu + E_i)/kT)
    acc   = (N / L**2) * np.exp((-mu + E_i) / kT)
    if rng.random() < min(1.0, acc):
        return q_new, True
    return q, False


def mc_sweep(q, L, kT, mu, params, rng, f_disp=0.5):
    """
    One sweep = max(N, 1) move attempts.
    Each attempt is independently chosen:
      displacement  with probability f_disp
      insertion     with probability (1-f_disp)/2
      deletion      with probability (1-f_disp)/2
    Returns (q_new, stats_dict).
    """
    n_att = max(len(q), 1)
    stats = {'nd': 0, 'ni': 0, 'nl': 0, 'ad': 0, 'ai': 0, 'al': 0}
    f_ins = (1.0 - f_disp) / 2.0
    for _ in range(n_att):
        r = rng.random()
        if r < f_disp:
            q, acc = mc_displacement(q, L, kT, params, rng)
            stats['nd'] += 1;  stats['ad'] += int(acc)
        elif r < f_disp + f_ins:
            q, acc = mc_insertion(q, L, kT, mu, params, rng)
            stats['ni'] += 1;  stats['ai'] += int(acc)
        else:
            q, acc = mc_deletion(q, L, kT, mu, params, rng)
            stats['nl'] += 1;  stats['al'] += int(acc)
    return q, stats


print('GCMC move functions defined.')

## 6 — Chemical Potential Scan

Run short GCMC sweeps at each μ value and record ⟨N⟩.  
The crossing of the ⟨N⟩ = N_target line gives **μ***.

In [ ]:
from tqdm import tqdm 
mu_values   = [-2., -1., -0.5]#np.linspace(-12.0, -2.0, 11)
N_equil_sc  = 100_000    # equilibration sweeps per mu
N_avg_sc    = 1000   # averaging sweeps per mu

mean_N_sc = []
std_N_sc  = []

print(f'   mu      <N>    std_N')
print('   ' + '-'*25)
for mu_val in mu_values:
    q_sc  = q0.copy()
    rng_sc = np.random.default_rng(7)
    for _ in tqdm(range(N_equil_sc), desc='Equilibration : {}'.format(_)):
        q_sc, _ = mc_sweep(q_sc, L, kT, mu_val, params, rng_sc)
    Ns = []
    for _ in tqdm(range(N_avg_sc), desc='Averaging : {}'.format(_)):
        q_sc, _ = mc_sweep(q_sc, L, kT, mu_val, params, rng_sc)
        Ns.append(len(q_sc))
    mean_N_sc.append(np.mean(Ns))
    std_N_sc.append(np.std(Ns))
    print(f'   {mu_val:6.2f}   {mean_N_sc[-1]:6.1f}  {std_N_sc[-1]:.1f}')

mean_N_sc = np.array(mean_N_sc)
std_N_sc  = np.array(std_N_sc)

# Choose mu* as the value whose <N> is closest to N_init
mu_star_idx = int(np.argmin(np.abs(mean_N_sc - N_init)))
mu_star     = float(mu_values[mu_star_idx])
print(f'\nmu* = {mu_star:.2f}   (<N> = {mean_N_sc[mu_star_idx]:.1f}, target N = {N_init})')

fig, ax = plt.subplots(figsize=(7, 4))
ax.errorbar(mu_values, mean_N_sc, yerr=std_N_sc, fmt='o-', capsize=4, color='steelblue', label='<N> (GCMC)')
ax.axhline(N_init,  color='red',    ls='--', lw=1.5, label=f'Target N = {N_init}')
ax.axvline(mu_star, color='green',  ls=':',  lw=2.0, label=f'mu* = {mu_star:.2f}')
ax.axvline(mu_ideal, color='orange', ls='-.', lw=1.5, label=f'mu_ideal = {mu_ideal:.2f}')
ax.set_xlabel('Chemical potential mu / kT')
ax.set_ylabel('<N>')
ax.set_title('Equation of state: <N> vs mu')
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig('mu_scan.png', dpi=150)
plt.show()

## 7 — Production GCMC Run

Run at **μ*** (from scan above).  
Configurations are saved every `SAVE_INT` sweeps throughout **both** equilibration and production phases for later inspection.

In [ ]:
N_EQUIL  = 1_000_000    # equilibration sweeps
N_PROD   = 20_000   # production sweeps

SAVE_INT = 1000    # save a configuration snapshot every N sweeps

rng_pr = np.random.default_rng(0)
q_pr   = q0.copy()

# ── Time-series storage ───────────────────────────────────────────────────────
ts_sweep  = []   # sweep index (negative = equil)
ts_N      = []   # number of particles
ts_phase  = []   # 'eq' or 'prod'

ts_acc_d  = []   # per-sweep displacement acceptance rate
ts_acc_i  = []   # per-sweep insertion acceptance rate
ts_acc_l  = []   # per-sweep deletion acceptance rate

# ── Configuration snapshot storage ───────────────────────────────────────────
saved_configs = []   # list of (N, 3) arrays
saved_labels  = []   # human-readable label for each snapshot

# ── Energy storage (sampled every SAVE_INT production sweeps) ─────────────────
ts_E_data = []   # list of [prod_sweep, total_energy, N]


def record(sweep_idx, phase, q, stats):
    ts_sweep.append(sweep_idx)
    ts_N.append(len(q))
    ts_phase.append(phase)
    if stats['nd'] > 0: ts_acc_d.append(stats['ad'] / stats['nd'])
    if stats['ni'] > 0: ts_acc_i.append(stats['ai'] / stats['ni'])
    if stats['nl'] > 0: ts_acc_l.append(stats['al'] / stats['nl'])


# ── Initial snapshot ──────────────────────────────────────────────────────────
saved_configs.append(q_pr.copy())
saved_labels.append('initial')

# ─────────────────────────────────────────────────────────────────────────────
# Equilibration
# ─────────────────────────────────────────────────────────────────────────────
print('=== Equilibration ===')
for s in tqdm(range(N_EQUIL)):
    q_pr, stats = mc_sweep(q_pr, L, kT, mu_star, params, rng_pr)
    record(s - N_EQUIL, 'eq', q_pr, stats)
    if (s + 1) % SAVE_INT == 0:
        saved_configs.append(q_pr.copy())
        saved_labels.append(f'eq s={s+1}')
        # print(f'  eq  sweep {s+1:4d}: N = {len(q_pr):3d}')

# ─────────────────────────────────────────────────────────────────────────────
# Production
# ─────────────────────────────────────────────────────────────────────────────
print(f'\n=== Production  (mu* = {mu_star:.2f}) ===')
for s in tqdm(range(N_PROD)):
    q_pr, stats = mc_sweep(q_pr, L, kT, mu_star, params, rng_pr)
    record(s, 'prod', q_pr, stats)
    if (s + 1) % SAVE_INT == 0:
        E_now = total_energy_pbc(q_pr, L, params)
        ts_E_data.append([s + 1, E_now, len(q_pr)])
        saved_configs.append(q_pr.copy())
        saved_labels.append(f'prod s={s+1}')
        # print(f'  prod sweep {s+1:4d}: N = {len(q_pr):3d},  E/N = {E_now/max(len(q_pr),1):.2f}')

# ─────────────────────────────────────────────────────────────────────────────
ts_sweep = np.array(ts_sweep)
ts_N     = np.array(ts_N)
ts_phase = np.array(ts_phase)
ts_E_arr = np.array(ts_E_data) if ts_E_data else np.zeros((0, 3))  # (n, 3)

prod_N = ts_N[ts_phase == 'prod']
print(f'\nDone.  <N>_prod = {np.mean(prod_N):.1f} +/- {np.std(prod_N):.1f}')
print(f'Saved {len(saved_configs)} configuration snapshots.')

---
## 8 — PLOTTING: Configuration Snapshots

Blue circles = particle cores.  Red dots = patch A.  Green dots = patch B.

In [ ]:
def plot_configuration(q, L, ax, title='', r_c=1.0):
    """Draw patchy particle configuration on ax."""
    ax.set_xlim(-L/2 - 0.5, L/2 + 0.5)
    ax.set_ylim(-L/2 - 0.5, L/2 + 0.5)
    ax.set_aspect('equal')
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_title(title, fontsize=8, pad=2)
    # Periodic box
    box = plt.Rectangle((-L/2, -L/2), L, L, fill=False, edgecolor='dimgray', lw=1.2)
    ax.add_patch(box)
    if len(q) == 0:
        ax.text(0, 0, 'N=0', ha='center', va='center', fontsize=10, color='gray')
        return
    # Cores
    for xi, yi, _ in q:
        ax.add_patch(Circle((xi, yi), r_c,
                             facecolor='steelblue', edgecolor='navy',
                             lw=0.4, alpha=0.55, zorder=2))
    # Patches
    pA, pB = get_patches(q, params['r_patch'], params['opening_angle'])
    ax.scatter(pA[:, 0], pA[:, 1], s=10, c='red',        zorder=5)
    ax.scatter(pB[:, 0], pB[:, 1], s=10, c='limegreen',  zorder=5)
    # N label
    ax.text(0.03, 0.97, f'N={len(q)}', transform=ax.transAxes,
            va='top', ha='left', fontsize=7, color='black',
            bbox=dict(fc='white', alpha=0.6, pad=1, edgecolor='none'))


print('plot_configuration() defined.')

In [ ]:
# ── Grid of all saved snapshots ───────────────────────────────────────────────
nc    = len(saved_configs)
ncols = 5
nrows = (nc + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols,
                          figsize=(3.8 * ncols, 3.8 * nrows),
                          squeeze=False)
axes_flat = axes.flatten()

for k, (q_k, lbl_k) in enumerate(zip(saved_configs, saved_labels)):
    plot_configuration(q_k, L, axes_flat[k], title=lbl_k)

for k in range(nc, len(axes_flat)):
    axes_flat[k].set_visible(False)

plt.suptitle(
    f'GCMC snapshots  (mu*={mu_star:.2f}, E_AB={E_AB}, opening_angle=90 deg)',
    y=1.01, fontsize=11)
plt.tight_layout()
plt.savefig('gcmc_snapshots.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Initial / mid / final comparison ─────────────────────────────────────────
n  = len(saved_configs)
sel    = [0, n // 2, n - 1]
titles = ['Initial', f'Mid  ({saved_labels[n//2]})', f'Final ({saved_labels[-1]})']

fig, axes = plt.subplots(1, 3, figsize=(14, 5))
for ax, idx, ttl in zip(axes, sel, titles):
    plot_configuration(saved_configs[idx], L, ax, title=ttl)

leg_handles = [
    mpatches.Patch(facecolor='steelblue', edgecolor='navy', alpha=0.6, label='Core'),
    plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='red',       markersize=9, label='Patch A'),
    plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='limegreen', markersize=9, label='Patch B'),
]
fig.legend(handles=leg_handles, loc='lower center', ncol=3,
           bbox_to_anchor=(0.5, -0.06), fontsize=10)
plt.suptitle(
    f'Configuration evolution  (E_AB={E_AB}, opening_angle=90 deg, mu*={mu_star:.2f})',
    fontsize=11)
plt.tight_layout()
plt.savefig('gcmc_evolution.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 9 — PLOTTING: Time Series

In [ ]:
eq_mask   = (ts_phase == 'eq')
prod_mask = (ts_phase == 'prod')

fig, axes = plt.subplots(2, 1, figsize=(12, 7))

# ── N(t) ─────────────────────────────────────────────────────────────────────
ax = axes[0]
ax.plot(ts_sweep[eq_mask],   ts_N[eq_mask],   color='gray',      lw=0.7, label='Equil.')
ax.plot(ts_sweep[prod_mask], ts_N[prod_mask], color='steelblue', lw=0.8, label='Prod.')
ax.axhline(N_init,            color='red',    ls='--', lw=1.5, label=f'Target N={N_init}')
ax.axhline(np.mean(prod_N),   color='orange', ls=':',  lw=1.5,
           label=f'<N>={np.mean(prod_N):.1f} +/- {np.std(prod_N):.1f}')
ax.axvline(0, color='black', lw=0.8, alpha=0.4, label='prod start')
ax.set_ylabel('N  (particles)')
ax.set_title('Particle number vs MC sweep')
ax.legend(fontsize=8, ncol=3)

# ── E/N over production ───────────────────────────────────────────────────────
ax2 = axes[1]
if len(ts_E_arr) > 0:
    e_sw  = ts_E_arr[:, 0]
    e_val = ts_E_arr[:, 1]
    e_N   = ts_E_arr[:, 2]
    ax2.plot(e_sw, e_val / np.maximum(e_N, 1), color='coral', lw=0.9)
    ax2.set_xlabel('Production sweep')
    ax2.set_ylabel('E / N  (kT)')
    ax2.set_title('Energy per particle (production, sampled every SAVE_INT sweeps)')
else:
    ax2.set_title('No energy data recorded.')

plt.tight_layout()
plt.savefig('gcmc_timeseries.png', dpi=150)
plt.show()

---
## 10 — PLOTTING: Acceptance Rates and N Distribution

In [ ]:
def rolling(arr, w):
    if len(arr) < w:
        return np.array(arr)
    return np.convolve(arr, np.ones(w) / w, mode='valid')


fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

# ── Acceptance rates (rolling average) ───────────────────────────────────────
W = 50
if len(ts_acc_d) > W: ax1.plot(rolling(ts_acc_d, W), label='Displacement', color='steelblue')
if len(ts_acc_i) > W: ax1.plot(rolling(ts_acc_i, W), label='Insertion',    color='green')
if len(ts_acc_l) > W: ax1.plot(rolling(ts_acc_l, W), label='Deletion',     color='red')
ax1.set_ylim(0, 1)
ax1.set_xlabel('Sweep')
ax1.set_ylabel('Acceptance rate')
ax1.set_title(f'MC acceptance rates (rolling mean, w={W})')
ax1.legend()

# ── P(N) histogram ───────────────────────────────────────────────────────────
bins = range(int(prod_N.min()), int(prod_N.max()) + 2)
ax2.hist(prod_N, bins=bins, density=True,
         color='steelblue', edgecolor='navy', alpha=0.7)
ax2.axvline(np.mean(prod_N), color='red',    ls='--', lw=1.5,
            label=f'<N> = {np.mean(prod_N):.1f}')
ax2.axvline(N_init,          color='orange', ls=':',  lw=2.0,
            label=f'Target = {N_init}')
ax2.set_xlabel('N')
ax2.set_ylabel('P(N)')
ax2.set_title('Particle-number distribution P(N)  [production]')
ax2.legend(fontsize=9)

plt.tight_layout()
plt.savefig('gcmc_stats.png', dpi=150)
plt.show()

# ── Summary statistics ────────────────────────────────────────────────────────
print(f'Production N:  mean={np.mean(prod_N):.1f},  std={np.std(prod_N):.2f},  '
      f'min={prod_N.min()},  max={prod_N.max()}')
if len(ts_acc_d) > 0: print(f'Disp acceptance:  {np.mean(ts_acc_d):.3f}')
if len(ts_acc_i) > 0: print(f'Ins  acceptance:  {np.mean(ts_acc_i):.3f}')
if len(ts_acc_l) > 0: print(f'Del  acceptance:  {np.mean(ts_acc_l):.3f}')

---
## 11 — Widom Test-Particle Insertion

Estimates μ_excess by randomly inserting **ghost** (non-interacting) test particles
and averaging the Boltzmann factor of their interaction energy:

$$\mu_{\rm exc} = -k_BT \ln\!\left\langle e^{-\Delta U_{\rm test}/k_BT} \right\rangle_{NVT}$$

$$\mu_{\rm total} = k_BT\ln(\rho_N) + \mu_{\rm exc}$$

This should agree with the μ* found by the scan above.

In [ ]:
n_widom  = 5000
rng_wid  = np.random.default_rng(999)
q_wid    = q_pr   # use final production configuration

bf = []
for _ in range(n_widom):
    xw  = rng_wid.uniform(-L/2, L/2)
    yw  = rng_wid.uniform(-L/2, L/2)
    thw = rng_wid.uniform(0.0, 2.0 * np.pi)
    q_t = np.vstack([q_wid, [[xw, yw, thw]]])
    dU  = particle_energy(len(q_wid), q_t, L, params)
    bf.append(np.exp(-dU / kT))

bf_mean   = np.mean(bf)
mu_exc    = -kT * np.log(bf_mean)
rho_now   = len(q_wid) / L**2
mu_id_now = kT * np.log(rho_now)
mu_widom  = mu_id_now + mu_exc

print(f'Widom insertion  ({n_widom} test particles):')
print(f'  rho_N at final config  = {rho_now:.5f}')
print(f'  mu_ideal (at rho_now)  = {mu_id_now:.3f} kT')
print(f'  mu_excess (Widom)      = {mu_exc:.3f} kT')
print(f'  mu_total  (Widom)      = {mu_widom:.3f} kT')
print(f'  mu*  used in sim       = {mu_star:.3f} kT')
print(f'  mu_ideal (ideal gas)   = {mu_ideal:.3f} kT')